# Non-Ergodic Mess3 Transformer: Results

This notebook loads the experiment results and generates all figures for the writeup.

**Run `experiments/run_experiment.py` first to generate data, train, and analyze.**

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

DATA_DIR = Path('../data')
FIG_DIR = Path('../figures')
RESULTS_DIR = Path('../results')
CKPT_DIR = Path('../checkpoints')
FIG_DIR.mkdir(exist_ok=True)

## 1. Dataset Overview

In [ ]:
from src.data.mess3 import build_default_components, compute_synchronisation_horizon, COMPONENT_PARAMS
from src.data.belief_update import initial_belief
from src.analysis.pca import compute_msp_attractor, simplex_to_2d

components = build_default_components()
n_star = compute_synchronisation_horizon(components)
print(f'Synchronisation horizon N* = {n_star:.2f}')

# Plot MSP fractals for each component
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['steelblue', 'darkorange', 'forestgreen']

for k, (comp, params, ax, color) in enumerate(zip(components, COMPONENT_PARAMS, axes, colors)):
    attractor = compute_msp_attractor(comp, n_iterations=30000)
    xy = simplex_to_2d(attractor)
    ax.scatter(xy[:, 0], xy[:, 1], s=0.3, c=color, alpha=0.4)
    ax.set_title(f'Component {params.name}\nα={params.alpha}, x={params.x}')
    ax.set_aspect('equal')
    ax.set_xlabel('Simplex dim 1')
    ax.set_ylabel('Simplex dim 2')

fig.suptitle('Ground-Truth MSP Fractals for Each Mess3 Component', y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / 'msp_fractals.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Load Results

In [ ]:
with open(RESULTS_DIR / 'summary.json') as f:
    summary = json.load(f)

with open(CKPT_DIR / 'history.json') as f:
    history = json.load(f)

print(f"N* = {summary['n_star']:.2f}")
print(f"Final π R² = {summary['pi_r2_final']:.4f}")
print(f"Final η R² = {summary['eta_r2_final']}")
print(f"PCA dims by layer: {summary['pca_dims_by_layer']}")

## 3. Training Loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

steps = history['steps']
axes[0].plot(steps, history['train_loss'], label='Train', alpha=0.8)
axes[0].plot(steps, history['val_loss'], label='Val', alpha=0.8)
axes[0].set_xlabel('Training step')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Per-position loss at final step
final_per_pos = np.array(history['per_position_loss'][-1])
axes[1].plot(range(1, len(final_per_pos) + 1), final_per_pos, 'o-', ms=5)
axes[1].axvline(summary['n_star'], color='red', linestyle='--', label=f'N* = {summary["n_star"]:.1f}')
axes[1].set_xlabel('Context position')
axes[1].set_ylabel('Cross-entropy loss')
axes[1].set_title('Per-Position Loss (Final Model)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / 'training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. PCA Dimensionality

In [ ]:
from PIL import Image
img = Image.open(FIG_DIR / 'pca_explained_variance.png')
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. R² by Layer and Position

In [ ]:
pi_r2_vs_pos = np.array(summary['pi_r2_vs_position'])   # (n_layers, L)
eta_r2_vs_pos = np.array(summary['eta_r2_vs_position']) # (n_layers, L, K)
n_layers, L, K = eta_r2_vs_pos.shape
positions = np.arange(L)

fig, axes = plt.subplots(1, n_layers, figsize=(7 * n_layers, 5))
if n_layers == 1:
    axes = [axes]

colors = ['steelblue', 'darkorange', 'forestgreen']
comp_names = [p.name for p in COMPONENT_PARAMS]

for layer_idx, ax in enumerate(axes):
    ax.plot(positions + 1, pi_r2_vs_pos[layer_idx], 'k-o', ms=5, lw=2, label='π')
    for k in range(K):
        ax.plot(positions + 1, eta_r2_vs_pos[layer_idx, :, k],
                '-s', color=colors[k], ms=4, lw=1.5, label=f'η_{k} ({comp_names[k]})')
    ax.axvline(summary['n_star'], color='red', linestyle='--', alpha=0.7,
               label=f'N* = {summary["n_star"]:.1f}')
    ax.set_xlabel('Context position')
    ax.set_ylabel('R²')
    ax.set_title(f'Layer {layer_idx}')
    ax.legend(fontsize=9)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)

fig.suptitle('R² vs Context Position: π and η_k Linear Decodability')
fig.tight_layout()
fig.savefig(FIG_DIR / 'r2_vs_position_all_layers.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Subspace Overlap Matrix

In [ ]:
overlap_matrix = np.array(summary['overlap_matrix'])
overlap_names = summary['overlap_names']

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(overlap_matrix, vmin=0, vmax=1, cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, label='Subspace overlap')
ax.set_xticks(range(len(overlap_names)))
ax.set_yticks(range(len(overlap_names)))
ax.set_xticklabels(overlap_names, rotation=45, ha='right')
ax.set_yticklabels(overlap_names)
for i in range(len(overlap_names)):
    for j in range(len(overlap_names)):
        ax.text(j, i, f'{overlap_matrix[i, j]:.2f}', ha='center', va='center',
                color='white' if overlap_matrix[i, j] > 0.5 else 'black', fontsize=9)
ax.set_title('Subspace Overlap Matrix\n(0=orthogonal, 1=identical)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'subspace_overlap.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Projection Tests (H2 vs H1)

In [ ]:
print('Projection Tests: Remove π subspace, check η_k R²')
print('=' * 50)
for k, pt in enumerate(summary['projection_tests']):
    verdict = 'H2 (factored)' if pt['r2_drop'] < 0.1 else 'H1 (joint)'
    print(f'  η_{k} ({comp_names[k]}): '
          f'R²_before={pt["r2_before"]:.3f}, '
          f'R²_after={pt["r2_after"]:.3f}, '
          f'drop={pt["r2_drop"]:.3f} → {verdict}')

## 8. Fractal Geometry

In [ ]:
from PIL import Image
img = Image.open(FIG_DIR / 'fractal_comparison.png')
plt.figure(figsize=(15, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Residual Stream Projection vs Ground-Truth MSP Fractal')
plt.show()

## 9. Training Dynamics

In [ ]:
dyn = summary['training_dynamics']
steps = np.array(dyn['steps'])
pi_r2 = np.array(dyn['pi_r2'])
eta_r2 = np.array(dyn['eta_r2'])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, pi_r2, 'k-o', ms=5, lw=2, label='π (meta-belief)')
for k in range(K):
    ax.plot(steps, eta_r2[:, k], '-s', color=colors[k], ms=4, lw=1.5,
            label=f'η_{k} ({comp_names[k]})')

if dyn['pi_transition_step']:
    ax.axvline(dyn['pi_transition_step'], color='k', linestyle=':', alpha=0.7,
               label=f'π transition @ {dyn["pi_transition_step"]}')

ax.set_xlabel('Training step')
ax.set_ylabel('R²')
ax.set_title('Belief Subspace Emergence over Training (Last Layer, Last Position)')
ax.legend(fontsize=9)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / 'training_dynamics_nb.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'π phase transition: {dyn["pi_transition_step"]}')
for k in range(K):
    print(f'η_{k} phase transition: {dyn["eta_transition_steps"][k]}')

## 10. Summary: Which Hypothesis?

Based on the results above, we evaluate the three competing hypotheses:

- **H1 (Joint)**: π and η_k share one undifferentiated subspace
- **H2 (Factored)**: K+1 orthogonal subspaces (1 for π, K for η_k)
- **H3 (Superposition)**: non-orthogonal but approximately recoverable

In [ ]:
print('HYPOTHESIS EVALUATION')
print('=' * 60)
print(f'π R² (should be ~1 for all): {summary["pi_r2_final"]:.4f}')
for k in range(K):
    print(f'η_{k} R² (should be ~1 for all): {summary["eta_r2_final"][k]:.4f}')
print()

max_off_diag = 0
names = summary['overlap_names']
om = np.array(summary['overlap_matrix'])
for i in range(len(names)):
    for j in range(len(names)):
        if i != j:
            max_off_diag = max(max_off_diag, om[i, j])

print(f'Max off-diagonal subspace overlap: {max_off_diag:.3f}')
print(f'  (near 0 = factored, near 1 = joint)')
print()

max_drop = max(pt['r2_drop'] for pt in summary['projection_tests'])
print(f'Max R² drop after removing π subspace: {max_drop:.3f}')
print(f'  (near 0 = factored H2, large = joint H1)')
print()

if max_off_diag < 0.2 and max_drop < 0.1:
    verdict = 'H2 (Factored): subspaces are approximately orthogonal'
elif max_off_diag > 0.5:
    verdict = 'H1 (Joint): subspaces overlap significantly'
else:
    verdict = 'H3 (Superposition): partial overlap, approximately factored'
print(f'VERDICT: {verdict}')